# LLM ocenjivanje relevantnosti query–product parova

Za svaki par koji čine korisnički upit i jedan proizvod, model dobija:
- tekst upita
- naziv, brend i kategoriju proizvoda
- opis proizvoda
- pros, cons i best_uses

Model svakom paru dodeljuje ocenu relevantnosti 0 - 3 i kratko obrazloženje ocene.

## 1. Učitavanje biblioteka

In [ ]:
import random
import numpy as np
from huggingface_hub import model_info
from transformers import AutoModelForCausalLM, AutoTokenizer
from pathlib import Path
import torch
import sys
import json
import re
import pandas as pd

## 2. Konfiguracija

In [2]:
if torch.cuda.is_available():
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path("/content/drive/MyDrive/RecSys")
else:
    PROJECT_ROOT = Path.cwd().parent
print('PROJECT_ROOT:', PROJECT_ROOT.resolve())
sys.path.insert(0, str(PROJECT_ROOT))

Mounted at /content/drive
PROJECT_ROOT: /content/drive/MyDrive/RecSys


In [ ]:
SEED = 42
MAX_NEW_TOKENS = 256
JUDGE_MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda'  if torch.cuda.is_available() else 'cpu'
EVAL_DIR = PROJECT_ROOT / 'data' / 'evaluation'
TEST_POOL_PATH = EVAL_DIR / 'llm_test_pool.csv' #Model je prvo pokrenut nad llm_test_pool.csv kako bi se korigovao prompt. 
POOL_PATH = EVAL_DIR / 'judge_pool.csv'
RESULT_PATH = EVAL_DIR / 'qrels.csv'

input_pool = pd.read_csv(POOL_PATH).fillna('')

print('Test pairs:', len(input_pool))
print('Queries per type:')
display(input_pool['query_type'].value_counts().sort_index())

Test pairs: 489
Queries per type:


,count
query_type,
broad_or_underspecified,103
multiple_requirements,127
problem_or_negative_constraint,156
single_property,103


Pool obuhvata različite tipove upita kako bi se proverilo da li llm razume jednostavne, višestruke, negativne i široko formulisane potrebe.

## 3. Definisanje skale relevantnosti i sistemskog prompta


Za ocene se koristi ordinalna skala:
- 0 - nerelevantan: pogrešan proizvod ili funkcija, nema dokaza za centralnu potrebu ili joj dokazi direktno protivreče
- 1 - delimično relevantan: tip proizvoda je približno odgovarajući, ali je centralni ili dodatni zahtev nepodržan, sporan ili direktno kontradiktoran
- 2 - relevantan: centralna potreba je podržana, ali dodatni eksplicitni zahtev nedostaje ili je neizvestan
- 3 - veoma relevantan: svi eksplicitni zahtevi su potvrđeni i ne postoji kontradikcija

In [4]:
SYSTEM_PROMPT = """You are a conservative relevance assessor for a cosmetics product retrieval system.

Judge exactly one query-product pair using only the supplied query and product evidence. Do not use outside knowledge, ratings, popularity, retrieval rank, similarity, or model identity. Do not infer medical or allergy safety.

First identify every explicit query requirement, including product type, function, finish, shade, skin type, coverage, durability, use case, price, brand, and negative constraints.

Use exactly this ordinal scale:
0 = Irrelevant: wrong product or function, no meaningful evidence for the central need, or the central need is clearly contradicted with credible supporting evidence.
1 = Marginally relevant: the product type is approximately suitable, but the central requirement is unsupported, disputed, or contradicted; also use 1 when an otherwise suitable product directly contradicts an explicit additional requirement.
2 = Relevant: the central need is supported, but an explicit additional requirement is missing or uncertain.
3 = Highly relevant: every explicit requirement is affirmatively supported and there is no query-relevant contradiction.

Evidence rules:
- Missing or uncertain evidence does not satisfy a requirement.
- Absence of evidence is not automatically a contradiction.
- Consider both pros and cons before assigning the grade.
- A direct query-relevant statement in cons is evidence and must not be ignored.
- If pros and cons directly conflict about the same explicit requirement, the grade cannot be 3. Normally use 1 because the requirement is disputed.
- Ignore negative evidence unrelated to the explicit query. For example, price complaints do not matter unless price is requested.
- Do not treat implied, typical, or common product properties as confirmed evidence.
- Grade 3 is allowed only when all explicit requirements have supplied affirmative evidence and none has relevant contradictory evidence.
- If any requirement is described as missing, uncertain, partial, unsupported, or contradicted in the justification, do not assign grade 3.
- A contradiction exists only when supplied evidence directly states or clearly demonstrates the opposite of an explicit query requirement.
- Do not infer a contradiction from a loosely related property or side effect.
- Missing, indirect, or uncertain evidence must be described as missing or uncertain, never as contradictory.
- Consider a negative item from cons only when it directly concerns an explicit query requirement or the basic requested product function.

Return exactly one valid JSON object with no markdown or additional text.
The justification must contain at most 60 words and at most two short sentences. Mention the decisive supporting evidence and any query-relevant contradiction.

Required format:
{"relevance": 0, "justification": "Concise evidence-based explanation."}
"""

## 4. Formiranje poruke za sudiju

In [5]:
def build_messages(row) :
    judging = {
        "query": row["query_text"],
        "product": {
            "name": row["product_name"],"brand": row["brand"],"category": row["category"],
            "description": row["description"],
            "pros": row["pros"],"cons": row["cons"],"best_uses": row["best_uses"]}}
    return[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user","content":"Assess this pair:\n"+json.dumps(judging, ensure_ascii=False)}
    ]

In [6]:
preview_messages = build_messages(input_pool.iloc[0].to_dict())
print('SYSTEM:')
print(preview_messages[0]['content'])
print('USER:')
print(preview_messages[1]['content'])

SYSTEM:
You are a conservative relevance assessor for a cosmetics product retrieval system.

Judge exactly one query-product pair using only the supplied query and product evidence. Do not use outside knowledge, ratings, popularity, retrieval rank, similarity, or model identity. Do not infer medical or allergy safety.

First identify every explicit query requirement, including product type, function, finish, shade, skin type, coverage, durability, use case, price, brand, and negative constraints.

Use exactly this ordinal scale:
0 = Irrelevant: wrong product or function, no meaningful evidence for the central need, or the central need is clearly contradicted with credible supporting evidence.
1 = Marginally relevant: the product type is approximately suitable, but the central requirement is unsupported, disputed, or contradicted; also use 1 when an otherwise suitable product directly contradicts an explicit additional requirement.
2 = Relevant: the central need is supported, but an exp

### Rezultat pregleda prompta

Prikazani sadržaj predstavlja tačan ulaz koji će model dobiti za svaki query–product par. Menjaju se samo konkretan upit i podaci proizvoda, dok sistemska pravila ostaju ista.

## 5. Učitavanje Qwen judge modela

In [7]:
MODEL_REVISION = model_info(JUDGE_MODEL_ID).sha

tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL_ID, revision=MODEL_REVISION)
judge_model = AutoModelForCausalLM.from_pretrained(JUDGE_MODEL_ID, revision=MODEL_REVISION,torch_dtype=torch.float16, device_map='auto', low_cpu_mem_usage=True,)
judge_model.eval()

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
        (post_attention_layer

## 6. Generisanje ocena relevantnosti

Sudija mora da vrati tačno jedan JSON objekat sa dve vrednosti
{
  "relevance": 0,
  "justification": "Concise evidence-based explanation."
}
Nevalidan format se beleži kao greška

In [ ]:
def parse_judgment(response):
    try:
        payload = json.loads(response.strip())
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid format detected : {e}")
    if not isinstance(payload, dict):
        raise ValueError("Judge response must be a JSON object.")

    relevance = payload.get("relevance")
    justification = payload.get("justification")
    if type(relevance) is not int or relevance not in {0, 1, 2, 3}:
        raise ValueError("Relevance must be an integer from 0 to 3.")
    if not isinstance(justification, str):
        raise ValueError("Justification must be a string.")
    justification = re.sub(r"\s+"," ",justification,).strip()
    if len(justification.split()) > 60:
        raise ValueError("Justification must contain at most 60 words")
    if not justification or len(justification) > 600:
        raise ValueError("Justification must contain 1 to 600 characters.")

    return relevance, justification

In [ ]:
def judge_pair(row):
    messages = build_messages(row)
    encoded = tokenizer.apply_chat_template(messages,add_generation_prompt=True, tokenize=True,return_dict=True,return_tensors="pt")
    encoded["input_ids"] = encoded["input_ids"].to(judge_model.device)
    encoded["attention_mask"] = encoded["attention_mask"].to(judge_model.device)
    input_length = encoded["input_ids"].shape[1]

    with torch.inference_mode():
        generated = judge_model.generate(**encoded,do_sample=False,max_new_tokens=MAX_NEW_TOKENS,pad_token_id=tokenizer.eos_token_id)

    raw_response = tokenizer.decode(generated[0, input_length:],skip_special_tokens=True).strip()

    result = {"query_id": row["query_id"],"logical_product_id": row["logical_product_id"],"query_text": row["query_text"],
            "query_type": row["query_type"],"product_name": row["product_name"],"raw_response": raw_response}

    try:
        parsed = parse_judgment(raw_response)

        result["relevance"] = parsed[0]
        result["justification"] = parsed[1]
        result["parse_status"] = "ok"

    except ValueError as error:
        result["relevance"] = pd.NA
        result["justification"] = str(error)
        result["parse_status"] = "error"

    return result

In [ ]:
results = pd.DataFrame()
completed = set()

for row in input_pool.to_dict(orient="records"):
    key = (str(row["query_id"]),str(row["logical_product_id"]))
    if key in completed:
        continue
    result = judge_pair(row)
    results = pd.concat([results,pd.DataFrame([result])],ignore_index=True)
    completed.add(key)
    print(f"{len(completed)}/{len(input_pool)} | status={result['parse_status']} | relevance={result['relevance']}")

results.to_csv(RESULT_PATH, index=False)

1/489 | status=ok | relevance=3
2/489 | status=ok | relevance=3
3/489 | status=ok | relevance=2
4/489 | status=ok | relevance=1
5/489 | status=ok | relevance=1
6/489 | status=ok | relevance=3
7/489 | status=ok | relevance=3
8/489 | status=ok | relevance=3
9/489 | status=ok | relevance=3
10/489 | status=ok | relevance=2
11/489 | status=ok | relevance=3
12/489 | status=ok | relevance=1
13/489 | status=ok | relevance=3
14/489 | status=ok | relevance=3
15/489 | status=ok | relevance=1
16/489 | status=ok | relevance=3
17/489 | status=ok | relevance=1
18/489 | status=ok | relevance=3
19/489 | status=ok | relevance=1
20/489 | status=ok | relevance=3
21/489 | status=ok | relevance=3
22/489 | status=ok | relevance=2
23/489 | status=ok | relevance=3
24/489 | status=ok | relevance=2
25/489 | status=ok | relevance=2
26/489 | status=ok | relevance=3
27/489 | status=ok | relevance=1
28/489 | status=ok | relevance=3
29/489 | status=ok | relevance=1
30/489 | status=ok | relevance=2
31/489 | status=ok 

## 7.Pregled ocena

Proverava se da:
- svaki odgovor ima parse_status="ok"
- sve relevance ocene pripadaju skupu 0–3
- relevance kolona može da se pretvori u celobrojni tip bez greške

Kod manjeg test podskupa radi modifikacije prompta je ručno provereno da li:
- model dosledno koristi skalu 0–3
- koristi samo prosleđene dokaze
- izbegava halucinacije
- poštuje eksplicitne negativne zahteve
- relevantne kontradikcije iz cons utiču na ocenu
- obrazloženje odgovara dodeljenoj oceni

In [11]:
results = pd.read_csv(RESULT_PATH, dtype={'query_id': 'string', 'logical_product_id': 'string'})
assert results['parse_status'].eq('ok').all(), ('At least one output is invalid. Inspect raw_response')
assert results['relevance'].between(0, 3).all()
results['relevance'] = pd.to_numeric(results['relevance'], errors='raise').astype(int)

print('Relevance distribution:')
display(results['relevance'].value_counts().sort_index())
print('Grades by query type:')
display(pd.crosstab(results['query_type'], results['relevance']))


Relevance distribution:


,count
relevance,
0,17
1,142
2,110
3,220


Grades by query type:


relevance,0,1,2,3
query_type,,,,
broad_or_underspecified,4,8,31,60
multiple_requirements,3,38,31,55
problem_or_negative_constraint,9,84,27,36
single_property,1,12,21,69
